<a href="https://colab.research.google.com/github/yc-115/programing-language/blob/main/%E3%80%8CHW3_%E5%BE%85%E8%BE%A6%E6%B8%85%E5%96%AE%E8%88%87%E7%95%AA%E8%8C%84%E9%90%98%E7%B4%80%E9%8C%84_ipynb%E3%80%8D41371211H.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -U -q google-generativeai
!pip -q install gspread gspread_dataframe google-auth google-auth-oauthlib google-auth-httplib2 \
                gradio pandas beautifulsoup4 google-generativeai python-dateutil
import os, time, uuid, re, json, datetime
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import pandas as pd
import gradio as gr
import requests
from bs4 import BeautifulSoup

import google.generativeai as genai

# Google Auth & Sheets
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from google.auth import default
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-2.5-pro')
SHEET_URL = "https://docs.google.com/spreadsheets/d/1MXEOE5i8pmdL5DiTr5twg68aDe9kq-Fh8uVvh75WPR4/edit?usp=sharing"
WORKSHEET_NAME = "工作表3"
TIMEZONE = "Asia/Taipei"
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url(SHEET_URL)


# 從 gsheets 的 All-whiteboard-device 載入 sheets
sh = gsheets.worksheet(WORKSHEET_NAME).get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sh[1:], columns=sh[0])
# 取得最前面的5筆資料
df.head()

def ensure_spreadsheet(name):
    try:
        sh = gc.open(name)  # returns gspread.models.Spreadsheet
    except gspread.SpreadsheetNotFound:
        sh = gc.create(name)
    return sh

sh = ensure_spreadsheet(WORKSHEET_NAME)
def ensure_spreadsheet(name):
    try:
        sh = gc.open(name)  # returns gspread.models.Spreadsheet
    except gspread.SpreadsheetNotFound:
        sh = gc.create(name)
    return sh

# Initialize the main spreadsheet object using its URL
# This is the correct spreadsheet where data should be stored
main_spreadsheet = gc.open_by_url(SHEET_URL)

def ensure_worksheet(sh, title, header):
    try:
        ws = sh.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = sh.add_worksheet(title=title, rows="1000", cols=str(len(header)+5))
        ws.update([header])
    # 若沒有表頭就補上
    data = ws.get_all_values()
    if not data or (data and data[0] != header):
        ws.clear()
        ws.update([header])
    return ws

TASKS_HEADER = [
    "id","task","status","priority","est_min","start_time","end_time",
    "actual_min","pomodoros","due_date","labels","notes",
    "created_at","updated_at","completed_at","planned_for"
]
LOGS_HEADER = [
    "log_id","task_id","phase","start_ts","end_ts","minutes","cycles","note"
]
CLIPS_HEADER = ["clip_id","url","selector","text","href","created_at","added_to_task"]

# Ensure specific worksheets exist within the main_spreadsheet object
ws_tasks = ensure_worksheet(main_spreadsheet, "tasks", TASKS_HEADER)
ws_logs  = ensure_worksheet(main_spreadsheet, "pomodoro_logs", LOGS_HEADER)
ws_clips = ensure_worksheet(main_spreadsheet, "web_clips", CLIPS_HEADER)

def tznow():
    return dt.now(gettz(TIMEZONE))

def read_df(ws, header):
    df = get_as_dataframe(ws, evaluate_formulas=True, header=0)
    if df is None or df.empty:
        return pd.DataFrame(columns=header)
    df = df.fillna("")
    # 保證欄位齊全
    for c in header:
        if c not in df.columns:
            df[c] = ""
    # 型別微調
    if "est_min" in df.columns:
        df["est_min"] = pd.to_numeric(df["est_min"], errors="coerce").fillna(0).astype(int)
    if "actual_min" in df.columns:
        df["actual_min"] = pd.to_numeric(df["actual_min"], errors="coerce").fillna(0).astype(int)
    if "pomodoros" in df.columns:
        df["pomodoros"] = pd.to_numeric(df["pomodoros"], errors="coerce").fillna(0).astype(int)
    return df[header]

def write_df(ws, df, header):
    # 清除原有內容並寫入DataFrame，使用gspread_dataframe更為可靠
    ws.clear()
    if not df.empty:
        set_with_dataframe(ws, df, include_column_header=True, row=1, col=1)
    else:
        ws.update([header]) # 如果df為空，只寫入標頭

def refresh_all():
    return (
        read_df(ws_tasks, TASKS_HEADER).copy(),
        read_df(ws_logs, LOGS_HEADER).copy(),
        read_df(ws_clips, CLIPS_HEADER).copy()
    )

tasks_df, logs_df, clips_df = refresh_all()

def add_task(task, priority, est_min, due_date, labels, notes, planned_for):
    global tasks_df, logs_df, clips_df
    _now = tznow().isoformat()
    new = pd.DataFrame([{
        "id": str(uuid.uuid4())[:8],
        "task": task.strip(),
        "status": "todo",
        "priority": priority or "M",
        "est_min": int(est_min) if est_min else 25,
        "start_time": "",
        "end_time": "",
        "actual_min": 0,
        "pomodoros": 0,
        "due_date": due_date or "",
        "labels": labels or "",
        "notes": notes or "",
        "created_at": _now,
        "updated_at": _now,
        "completed_at": "",
        "planned_for": planned_for or ""
    }])
    tasks_df = pd.concat([tasks_df, new], ignore_index=True)
    write_df(ws_tasks, tasks_df, TASKS_HEADER)

    # 確保資料刷新
    tasks_df, logs_df, clips_df = refresh_all()

    # 1. 產生新的下拉選單選項
    new_choices = list_task_choices()

    # 2. 必須回傳 4 個值，對應 btn_add.click 的 outputs
    # 分別是：訊息文字、DataFrame表格、更新後的選單1、更新後的選單2
    return (
        "✅ 已新增任務",
        tasks_df,
        gr.update(choices=new_choices),
        gr.update(choices=new_choices)
    )

def update_task_status(task_id, new_status):
    global tasks_df
    idx = tasks_df.index[tasks_df["id"] == task_id]
    if len(idx)==0:
        return "⚠️ 找不到任務", tasks_df
    i = idx[0]
    tasks_df.loc[i, "status"] = new_status
    tasks_df.loc[i, "updated_at"] = tznow().isoformat()
    if new_status == "done" and not tasks_df.loc[i, "completed_at"]:
        tasks_df.loc[i, "completed_at"] = tznow().isoformat()
    write_df(ws_tasks, tasks_df, TASKS_HEADER)

    # After writing, refresh all dataframes by re-reading from the sheet
    global logs_df, clips_df
    tasks_df, logs_df, clips_df = refresh_all()

    return "✅ 狀態已更新", tasks_df

def mark_done(task_id):
    return update_task_status(task_id, "done")

def recalc_task_actuals(task_id):
    """根據 logs_df 回寫 actual_min 與 pomodoros"""
    global tasks_df, logs_df
    work_logs = logs_df[(logs_df["task_id"]==task_id) & (logs_df["phase"]=="work")]
    total_min = work_logs["minutes"].astype(float).sum() if not work_logs.empty else 0
    pomos = int(round(total_min / 25.0))
    idx = tasks_df.index[tasks_df["id"]==task_id]
    if len(idx)==0: return
    i = idx[0]
    tasks_df.loc[i,"actual_min"] = int(total_min)
    tasks_df.loc[i,"pomodoros"] = pomos
    tasks_df.loc[i,"updated_at"] = tznow().isoformat()

def list_task_choices():
    global tasks_df
    if tasks_df.empty:
        return []
    # 顯示： [status] (P:priority) task  — id
    def row_label(r):
        return f"[{r['status']}] (P:{r['priority']}) {r['task']} — {r['id']}"
    return [(row_label(r), r["id"]) for _, r in tasks_df.iterrows()]

# 我們採「按鈕開始 / 結束」模式（避免後端阻塞），每次按「開始」會先記住 start_ts，
# 按「結束」時計算分鐘並寫入 logs，再回填任務 actual_min / pomodoros。

_active_sessions = {}  # { task_id: {"phase": "work"/"break", "start_ts": iso, "cycles": int} }

def start_phase(task_id, phase, cycles):
    if not task_id: return "⚠️ 請先選擇任務"
    _active_sessions[task_id] = {
        "phase": phase,
        "start_ts": tznow().isoformat(),
        "cycles": int(cycles) if cycles else 1
    }
    return f"▶️ 已開始：{phase}（task: {task_id}）"

def end_phase(task_id, note):
    global logs_df, tasks_df
    if task_id not in _active_sessions:
        return "⚠️ 尚未開始任何階段"
    sess = _active_sessions.pop(task_id)
    start = pd.to_datetime(sess["start_ts"])
    end = tznow()
    minutes = round((end - start).total_seconds() / 60.0, 2)
    log = pd.DataFrame([{
        "log_id": str(uuid.uuid4())[:8],
        "task_id": task_id,
        "phase": sess["phase"],
        "start_ts": start.isoformat(),
        "end_ts": end.isoformat(),
        "minutes": minutes,
        "cycles": int(sess["cycles"]),
        "note": note or ""
    }])
    logs_df = pd.concat([logs_df, log], ignore_index=True)
    write_df(ws_logs, logs_df, LOGS_HEADER)

    # 回填任務
    if sess["phase"] == "work":
        recalc_task_actuals(task_id)
        write_df(ws_tasks, tasks_df, TASKS_HEADER)

    # After writing, refresh all dataframes by re-reading from the sheet
    global clips_df
    tasks_df, logs_df, clips_df = refresh_all()

    return f"⏹️ 已結束：{sess['phase']}，紀錄 {minutes} 分鐘"

# AI 計畫（Gemini；無金鑰則規則式）
def generate_today_plan():
    global tasks_df
    # 以「due_date 是今天」或「planned_for = today」且不是 done 的任務為計畫清單
    today = tznow().date().isoformat()
    cand = tasks_df[
        ((tasks_df["due_date"]==today) | (tasks_df["planned_for"].str.lower()=="today")) &
        (tasks_df["status"]!="done")
    ].copy()
    if cand.empty:
        return "📭 今天沒有標記的任務。請在 Tasks 分頁把任務的 due_date 設為今天或 planned_for 設為 today。"

    # 先依 priority（H>M>L）+ est_min 排序
    pr_order = {"H":0, "M":1, "L":2}
    cand["p_ord"] = cand["priority"].map(pr_order).fillna(3)
    cand = cand.sort_values(["p_ord","est_min"], ascending=[True, True])

    # 嘗試 Gemini
    api_key = os.environ.get("GEMINI_API_KEY","").strip()
    if api_key:
        genai.configure(api_key=api_key)
        sys_prompt = (
            "你是一位任務規劃助理。請把輸入的任務（含估時與優先級）排成三段：morning、afternoon、evening，"
            "並給出每段的重點、順序、每項的時間預估與備註。總時數請大致符合任務估時總和。"
            "回傳以 Markdown 條列，格式：\n"
            "### Morning\n- [任務ID] 任務名稱（預估 xx 分）— 備註\n..."
            "### Afternoon\n...\n### Evening\n...\n"
        )
        items = []
        for _, r in cand.iterrows():
            items.append({
                "id": r["id"], "task": r["task"], "est_min": int(r["est_min"]),
                "priority": r["priority"]
            })
        user_content = json.dumps({"today": today, "tasks": items}, ensure_ascii=False)
        try:
            resp = model.generate_content(sys_prompt + "\n\n" + user_content)
            plan_md = resp.text
        except Exception as e:
            plan_md = f"⚠️ Gemini 失敗：{e}\n\n改用規則式規劃。"
    else:
        plan_md = "🔧 未設定 GEMINI_API_KEY，使用規則式規劃。\n\n"

    # 規則式：把高優先任務平均切到上午/下午/晚上
    buckets = {"morning": [], "afternoon": [], "evening": []}
    total = len(cand)
    for i, (_, r) in enumerate(cand.iterrows()):
        if i % 3 == 0:
            buckets["morning"].append(r)
        elif i % 3 == 1:
            buckets["afternoon"].append(r)
        else:
            buckets["evening"].append(r)

    def sec_md(name, rows):
        if not rows: return f"### {name.title()}\n（無）\n"
        lines = [f"### {name.title()}"]
        for r in rows:
            lines.append(f"- [{r['id']}] {r['task']}（預估 {int(r['est_min'])} 分，P:{r['priority']}）")
        return "\n".join(lines) + "\n"

    rule_md = (
        sec_md("morning", buckets["morning"]) + "\n" +
        sec_md("afternoon", buckets["afternoon"]) + "\n" +
        sec_md("evening", buckets["evening"])
    )

    return (plan_md + "\n---\n" + rule_md).strip()

def generate_rule_based_plan():
    global tasks_df
    # 1. 篩選今天的任務
    # 這裡增加一個判斷，確保 planned_for 不為空且為 today
    today_tasks = tasks_df[tasks_df["planned_for"].astype(str).str.lower() == "today"].copy()

    if today_tasks.empty:
        return "📭 今天還沒排定任務喔！請先到 Tasks 分頁將任務的 '規劃歸屬' 設為 'today' 並點擊更新。"

    # 2. 排序：優先級 H > M > L
    prio_map = {'H': 0, 'M': 1, 'L': 2}
    today_tasks['p_val'] = today_tasks['priority'].map(prio_map).fillna(1) # 若無優先級預設為 M
    today_tasks = today_tasks.sort_values('p_val')

    # 3. 模擬排程 (修正這裡：使用 dt.now 而不是 datetime.now)
    # 因為你在開頭已經 from datetime import datetime as dt
    current_time = dt.now(gettz(TIMEZONE)).replace(hour=9, minute=0, second=0, microsecond=0)

    plan_md = f"## 📅 今日硬核讀書計畫 ({current_time.strftime('%Y-%m-%d')})\n"
    plan_md += "| 時間 | 任務名稱 | 優先級 | 預估時間 |\n| --- | --- | --- | --- |\n"

    for _, row in today_tasks.iterrows():
        # 確保 est_min 是數字
        try:
            duration = int(row['est_min']) if row['est_min'] else 25
        except:
            duration = 25

        end_time = current_time + timedelta(minutes=duration)
        plan_md += f"| {current_time.strftime('%H:%M')} - {end_time.strftime('%H:%M')} | {row['task']} | {row['priority']} | {duration} min |\n"

        # 每個任務中間自動加 10 分鐘休息
        current_time = end_time + timedelta(minutes=10)

    return plan_md

# 今日完成率
def today_summary():
    global tasks_df
    today = tznow().date().isoformat()
    planned = tasks_df[
        ((tasks_df["due_date"]==today) | (tasks_df["planned_for"].str.lower()=="today"))
    ]
    done = planned[planned["status"]=="done"]
    total = len(planned)
    done_n = len(done)
    rate = (done_n/total*100) if total>0 else 0
    return f"📅 今日計畫任務：{total}；✅ 完成：{done_n}；📈 完成率：{rate:.1f}%"

# =========================
# 爬蟲：擷取文字或連結並可加入任務
# =========================
def crawl(url, selector, mode, limit):
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent":"Mozilla/5.0"})
        resp.raise_for_status()
    except Exception as e:
        return pd.DataFrame(columns=CLIPS_HEADER), f"⚠️ 請求失敗：{e}"

    soup = BeautifulSoup(resp.text, "html.parser")
    nodes = soup.select(selector)
    rows = []
    for i, n in enumerate(nodes[:int(limit) if limit else 20]):
        text = n.get_text(strip=True) if mode in ("text","both") else ""
        href = n.get("href") if mode in ("href","both") else ""
        # 相對連結處理
        if href and href.startswith("/"):
            from urllib.parse import urljoin
            href = urljoin(url, href)
        rows.append({
            "clip_id": str(uuid.uuid4())[:8],
            "url": url,
            "selector": selector,
            "text": text,
            "href": href,
            "created_at": tznow().isoformat(),
            "added_to_task": ""
        })
    df = pd.DataFrame(rows, columns=CLIPS_HEADER)
    return df, f"✅ 擷取 {len(df)} 筆"

def add_clips_as_tasks(clip_ids, default_priority, est_min):
    global clips_df, tasks_df
    if not clip_ids:
        return "⚠️ 請先勾選要加入的爬蟲項目", clips_df, tasks_df
    sel = clips_df[clips_df["clip_id"].isin(clip_ids)]
    _now = tznow().isoformat()
    new_tasks = []
    for _, r in sel.iterrows():
        title = r["text"] or r["href"] or "（未命名）"
        note = f"來源：{r['url']}\n選擇器：{r['selector']}\n連結：{r['href']}"
        new_tasks.append({
            "id": str(uuid.uuid4())[:8],
            "task": title[:120],
            "status": "todo",
            "priority": default_priority or "M",
            "est_min": int(est_min) if est_min else 25,
            "start_time": "",
            "end_time": "",
            "actual_min": 0,
            "pomodoros": 0,
            "due_date": "",
            "labels": "from:crawler",
            "notes": note,
            "created_at": _now,
            "updated_at": _now,
            "completed_at": "",
            "planned_for": ""
        })
    if new_tasks:
        tasks_df = pd.concat([tasks_df, pd.DataFrame(new_tasks)], ignore_index=True)
        # 標記已加入
        clips_df.loc[clips_df["clip_id"].isin(clip_ids), "added_to_task"] = "yes"
        write_df(ws_tasks, tasks_df, TASKS_HEADER)
        write_df(ws_clips, clips_df, CLIPS_HEADER)

        # After writing, refresh all dataframes by re-reading from the sheet
        global logs_df
        tasks_df, logs_df, clips_df = refresh_all()

        return f"✅ 已加入 {len(new_tasks)} 項為任務", clips_df, tasks_df
    return "⚠️ 無可加入項目", clips_df, tasks_df
import re

def analyze_course_announcement_v2(url):
    global tasks_df
    try:
        # 1. 爬取網頁內容
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, timeout=10, headers=headers)
        resp.encoding = 'utf-8'
        resp.raise_for_status()

        soup = BeautifulSoup(resp.text, "html.parser")
        title = soup.title.string.strip() if soup.title else "未命名公告"
        # 取得純文字並移除多餘空格
        text_content = soup.get_text(separator=" ", strip=True)

        # 2. 關鍵字過濾 (支援中英文)
        keywords = ["作業", "繳交", "截止", "測驗", "考試", "Homework", "Assignment", "Due", "Deadline", "Quiz", "Submit"]
        found_keywords = [k for k in keywords if k.lower() in text_content.lower()]

        if not found_keywords:
            return "ℹ️ 規則引擎判斷：此網頁可能不包含待辦任務。", tasks_df

        # 3. 正則表達式提取日期 (支援 2026/05/20, 05-20, 5月20日 等格式)
        date_pattern = r'(\d{4}[-/年]\d{1,2}[-/月]\d{1,2}日?|\d{1,2}[-/月]\d{1,2}日?)'
        dates = re.findall(date_pattern, text_content)
        due_date = dates[0] if dates else "" # 取第一個出現的日期作為截止日

        # 4. 簡單的優先級邏輯
        # 如果包含「考試」或「截止」，且有日期，設為 H
        priority = "M"
        if any(k in found_keywords for k in ["考試", "Deadline", "Quiz", "重要"]):
            priority = "H"
        elif len(found_keywords) <= 1:
            priority = "L"

        # 5. 避免重複加入
        # 檢查 notes 裡是否已經有這個網址
        if not tasks_df.empty:
            is_dup = tasks_df["notes"].str.contains(url).any()
            if is_dup:
                return "⚠️ 偵測到重複：此公告任務已存在於清單中。", tasks_df

        # 6. 產生任務資料
        _now = tznow().isoformat()
        new_item = {
            "id": str(uuid.uuid4())[:8],
            "task": f"📚 [規則偵測] {title[:30]}...",
            "status": "todo",
            "priority": priority,
            "est_min": 60,
            "due_date": due_date.replace("年","/").replace("月","/").replace("日",""), # 格式化日期
            "labels": "auto-course",
            "notes": f"偵測到關鍵字: {', '.join(found_keywords)}\n來源網址: {url}",
            "created_at": _now,
            "updated_at": _now,
            "planned_for": "today" if priority == "H" else ""
        }

        # 更新並寫入
        new_df = pd.DataFrame([new_item])
        tasks_df = pd.concat([tasks_df, new_df], ignore_index=True).fillna("")
        write_df(ws_tasks, tasks_df, TASKS_HEADER)

        # 刷新資料
        refresh_all()

        return f"✅ 規則引擎成功提取！關鍵字：{found_keywords[0]}...", tasks_df

    except Exception as e:
        return f"❌ 處理失敗：{str(e)}", tasks_df

# =========================
# Gradio 介面實作
# =========================
def _refresh():
    global tasks_df, logs_df, clips_df
    tasks_df, logs_df, clips_df = refresh_all()
    new_choices = list_task_choices()
    return (
        tasks_df,
        logs_df,
        clips_df,
        gr.update(choices=new_choices),
        gr.update(choices=new_choices),
        today_summary()
    )

with gr.Blocks(title="待辦清單＋番茄鐘＋AI 計畫系統") as demo:
    gr.Markdown("# ✅ 待辦清單與番茄鐘系統")

    with gr.Row():
        btn_refresh = gr.Button("🔄 重新整理（Sheet → App）")
        out_summary = gr.Markdown(today_summary())

    # --- 1. Tasks 分頁 ---
    with gr.Tab("Tasks"):
        with gr.Row():
            with gr.Column(scale=2):
                task_input = gr.Textbox(label="任務名稱", placeholder="輸入待辦事項...")
                priority_input = gr.Dropdown(["H","M","L"], value="M", label="優先級")
                est_input = gr.Number(value=25, label="預估時間 (min)", precision=0)
                due_input = gr.Textbox(label="到期日 (YYYY-MM-DD)")
                labels_input = gr.Textbox(label="標籤")
                notes_input = gr.Textbox(label="備註")
                planned_input = gr.Dropdown(["","today","tomorrow"], value="", label="規劃歸屬")
                btn_add = gr.Button("➕ 新增任務", variant="primary")
                msg_add = gr.Markdown()
            with gr.Column(scale=3):
                grid_tasks = gr.Dataframe(value=tasks_df, label="任務清單", interactive=False)

        with gr.Row():
            task_choice = gr.Dropdown(choices=list_task_choices(), label="選取任務（用於更新）")
            new_status = gr.Dropdown(["todo","in-progress","done"], value="in-progress", label="更新狀態")
            btn_update = gr.Button("✏️ 更新狀態")
            btn_done = gr.Button("✅ 直接標記完成")
            msg_update = gr.Markdown()

    # --- 2. Pomodoro 分頁 ---
    with gr.Tab("Pomodoro"):
        with gr.Row():
            sel_task = gr.Dropdown(choices=list_task_choices(), label="選擇任務")
            cycles = gr.Number(value=1, precision=0, label="番茄數")
        with gr.Row():
            btn_start_work = gr.Button("▶️ 開始工作", variant="primary")
            note_work = gr.Textbox(label="工作備註")
            btn_end_work = gr.Button("⏹️ 結束工作並記錄")
        with gr.Row():
            btn_start_break = gr.Button("🍵 開始休息")
            note_break = gr.Textbox(label="休息備註（可空白）")
            btn_end_break = gr.Button("⏹️ 結束休息並記錄")
        msg_pomo = gr.Markdown()
        grid_logs = gr.Dataframe(value=logs_df, label="番茄鐘紀錄")

    # --- 3. AI Plan 分頁 ---
    with gr.Tab("AI Plan"):
        gr.Markdown("### 🧠 讀書計畫產生器")
        with gr.Row():
            btn_plan_ai = gr.Button("AI 智慧排程 (Gemini)", variant="primary")
            btn_plan_rule = gr.Button("規則硬核排程 (穩定版)")
        out_plan = gr.Markdown()

    # --- 4. Crawler 分頁 ---
    with gr.Tab("Crawler"):
        url_input = gr.Textbox(label="目標 URL")
        selector_input = gr.Textbox(label="CSS Selector")
        mode_input = gr.Radio(["text","href","both"], value="text", label="擷取內容")
        limit_input = gr.Number(value=20, precision=0, label="限制筆數")
        btn_crawl = gr.Button("🕷️ 開始擷取")
        msg_crawl = gr.Markdown()
        grid_clips = gr.Dataframe(value=clips_df, label="擷取結果", interactive=True)
        clip_ids_input = gr.Textbox(label="要加入任務的 clip_id (逗號分隔)")
        btn_add_clips = gr.Button("➕ 加入任務")
        msg_add_clips = gr.Markdown()

    # --- 5. Course AI 分頁 ---
    with gr.Tab("🎓 Course AI"):
        gr.Markdown("### 課程公告自動化")
        with gr.Row():
            course_url_input = gr.Textbox(label="課程網址")
            btn_course_ai = gr.Button("🚀 智能解析", variant="primary")
        msg_course_ai = gr.Markdown()
        grid_course_auto = gr.Dataframe(label="自動產生的課程任務")

    # --- 6. Summary 分頁 ---
    with gr.Tab("Summary"):
        gr.Markdown("### 📊 數據統計與完成率")
        btn_summary_refresh = gr.Button("🔄 重新計算今日統計數據", variant="primary")
        out_summary_final = gr.Markdown()

    # =========================
    # 動作綁定 (Bindings)
    # =========================

    # 1. 刷新功能
    btn_refresh.click(_refresh, outputs=[grid_tasks, grid_logs, grid_clips, task_choice, sel_task, out_summary])

    # 2. 任務增刪改
    btn_add.click(add_task,
                  inputs=[task_input, priority_input, est_input, due_input, labels_input, notes_input, planned_input],
                  outputs=[msg_add, grid_tasks, task_choice, sel_task])

    btn_update.click(update_task_status, inputs=[task_choice, new_status], outputs=[msg_update, grid_tasks])
    btn_done.click(mark_done, inputs=[task_choice], outputs=[msg_update, grid_tasks])

    # 3. 番茄鐘邏輯
    btn_start_work.click(start_phase, inputs=[sel_task, gr.State("work"), cycles], outputs=[msg_pomo])
    btn_end_work.click(end_phase, inputs=[sel_task, note_work], outputs=[msg_pomo])
    btn_start_break.click(start_phase, inputs=[sel_task, gr.State("break"), cycles], outputs=[msg_pomo])
    btn_end_break.click(end_phase, inputs=[sel_task, note_break], outputs=[msg_pomo])

    # 4. 計畫產生
    btn_plan_ai.click(generate_today_plan, outputs=[out_plan])
    btn_plan_rule.click(generate_rule_based_plan, outputs=[out_plan])

    # 5. 爬蟲與解析
    def _handle_crawl(u, s, m, l):
        df, msg = crawl(u, s, m, l)
        global clips_df
        if not df.empty:
            clips_df = pd.concat([clips_df, df], ignore_index=True)
            write_df(ws_clips, clips_df, CLIPS_HEADER)
        return msg, clips_df

    btn_crawl.click(_handle_crawl, inputs=[url_input, selector_input, mode_input, limit_input], outputs=[msg_crawl, grid_clips])

    btn_course_ai.click(
        analyze_course_announcement_v2, # Changed from analyze_course_announcement to analyze_course_announcement_v2
        inputs=[course_url_input],
        outputs=[msg_course_ai, grid_tasks]
    ).then(
        # 解析完後，過濾出 labels 為 auto-course 的任務顯示在下方小表中
        lambda: tasks_df[tasks_df["labels"] == "auto-course"] if not tasks_df.empty else tasks_df,
        outputs=[grid_course_auto]
    )

    btn_add_clips.click(
        _add_clips,
        inputs=[clip_ids, default_priority, clip_est],
        outputs=[msg_add_clips, grid_clips, grid_tasks]
    )

    # 6. Summary
    btn_summary_refresh.click(today_summary, outputs=[out_summary_final])

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://421355395ff0ef5043.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
